# Playbook: Monthly Data Reconciliation

## Purpose
Perform monthly reconciliation between source systems and data warehouse to ensure data accuracy.

## Schedule
Run on the 1st of each month for previous month's data

## Success Criteria
- Row counts match within 0.1%
- No unexpected NULL values
- Date ranges complete
- Key metrics align with source

In [ ]:
# Setup
import os
import sys
os.environ['SPARK_HOME'] = '/opt/spark'
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, min, max, countDistinct
from datetime import datetime, timedelta
import pandas as pd

spark = SparkSession.builder \
    .appName("MonthlyReconciliation") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"✅ Monthly Reconciliation Started: {datetime.now()}")

In [ ]:
# PARAMETERS - Update for each reconciliation
RECONCILIATION_MONTH = "2024-02"  # Format: YYYY-MM
TABLES_TO_CHECK = ["customers", "orders"]  # Add all tables

# Calculate date range
from dateutil.relativedelta import relativedelta
from datetime import datetime

month_date = datetime.strptime(RECONCILIATION_MONTH, "%Y-%m")
start_date = month_date.strftime("%Y-%m-01")
end_date = (month_date + relativedelta(months=1, days=-1)).strftime("%Y-%m-%d")

print(f"Reconciling data for: {RECONCILIATION_MONTH}")
print(f"Date range: {start_date} to {end_date}")
print(f"Tables: {', '.join(TABLES_TO_CHECK)}")

## Step 1: Row Count Reconciliation

In [ ]:
# Compare row counts across layers
print("\n=== ROW COUNT RECONCILIATION ===")

results = []

for table in TABLES_TO_CHECK:
    counts = {}
    
    for layer in ["raw", "bronze", "silver", "gold"]:
        try:
            count_query = f"""
            SELECT COUNT(*) as cnt 
            FROM nessie.{layer}.{table}
            WHERE created_date >= '{start_date}' 
              AND created_date <= '{end_date}'
            """
            result = spark.sql(count_query)
            counts[layer] = result.collect()[0]['cnt']
        except:
            counts[layer] = None
    
    results.append({
        'Table': table,
        'Raw': counts.get('raw'),
        'Bronze': counts.get('bronze'),
        'Silver': counts.get('silver'),
        'Gold': counts.get('gold')
    })

recon_df = pd.DataFrame(results)
print(recon_df.to_string(index=False))

# Check for discrepancies
print("\n=== DISCREPANCY CHECK ===")
for _, row in recon_df.iterrows():
    if row['Raw'] != row['Bronze']:
        print(f"⚠️  {row['Table']}: Raw ({row['Raw']}) ≠ Bronze ({row['Bronze']})")
    elif row['Bronze'] != row['Silver']:
        print(f"⚠️  {row['Table']}: Bronze ({row['Bronze']}) ≠ Silver ({row['Silver']})")
    else:
        print(f"✅ {row['Table']}: All layers match")

## Step 2: Date Range Completeness

In [ ]:
# Check for missing dates
print("\n=== DATE RANGE COMPLETENESS ===")

for table in TABLES_TO_CHECK:
    print(f"\n{table.upper()}:")
    
    date_check = spark.sql(f"""
        SELECT 
            MIN(created_date) as earliest,
            MAX(created_date) as latest,
            COUNT(DISTINCT created_date) as distinct_dates
        FROM nessie.silver.{table}
        WHERE created_date >= '{start_date}' 
          AND created_date <= '{end_date}'
    """)
    
    date_check.show()

## Step 3: Key Metrics Validation

In [ ]:
# Validate key business metrics
print("\n=== KEY METRICS VALIDATION ===")

# Example: For orders table
if "orders" in TABLES_TO_CHECK:
    print("\nORDERS Metrics:")
    
    metrics = spark.sql(f"""
        SELECT 
            'Silver' as layer,
            COUNT(*) as total_orders,
            SUM(amount) as total_revenue,
            AVG(amount) as avg_order_value,
            COUNT(DISTINCT customer_id) as unique_customers
        FROM nessie.silver.orders
        WHERE created_date >= '{start_date}' 
          AND created_date <= '{end_date}'
        
        UNION ALL
        
        SELECT 
            'Gold' as layer,
            COUNT(*) as total_orders,
            SUM(amount) as total_revenue,
            AVG(amount) as avg_order_value,
            COUNT(DISTINCT customer_id) as unique_customers
        FROM nessie.gold.orders
        WHERE created_date >= '{start_date}' 
          AND created_date <= '{end_date}'
    """)
    
    metrics.show()

## Step 4: Data Quality Checks

In [ ]:
# Check for NULL values in critical columns
print("\n=== DATA QUALITY CHECKS ===")

for table in TABLES_TO_CHECK:
    print(f"\n{table.upper()} - NULL value check:")
    
    null_check = spark.sql(f"""
        SELECT 
            COUNT(*) as total_rows,
            SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) as null_customer_id,
            SUM(CASE WHEN created_date IS NULL THEN 1 ELSE 0 END) as null_created_date
        FROM nessie.silver.{table}
        WHERE created_date >= '{start_date}' 
          AND created_date <= '{end_date}'
    """)
    
    null_check.show()

## Step 5: Generate Reconciliation Report

In [ ]:
# Generate summary report
print("\n" + "="*60)
print(f"MONTHLY RECONCILIATION REPORT - {RECONCILIATION_MONTH}")
print("="*60)
print(f"\nRun Date: {datetime.now()}")
print(f"Period: {start_date} to {end_date}")
print(f"\nTables Reconciled: {len(TABLES_TO_CHECK)}")
print("\n" + "="*60)

# TODO: Add pass/fail criteria
print("\nSTATUS: ✅ PASSED (or ⚠️ REVIEW REQUIRED)")
print("\nIssues Found:")
print("- [List any issues here]")
print("\nRecommendations:")
print("- [List recommendations]")

## Sign-Off

- **Reconciled by**: 
- **Date**: 
- **Status**: ✅ Approved / ⚠️ Issues Found
- **Next reconciliation**: 